In [1]:
import numpy as np
from scipy.stats import norm

from engin_core.gp import fit_gp, split_conformal_multiplier
from engin_core.simulator import simulate_unit

GAUSS_90 = norm.ppf(0.95)   # ±1.645 sd, the textbook 90% Gaussian interval
NOMINAL = 0.90

def campaign(seed, n=120, d=5):
    """A space-filling DoE with heteroscedastic observation noise."""
    rng = np.random.default_rng(seed)
    U = rng.random((n, d))
    y_true = simulate_unit(U)
    y_obs = np.maximum(y_true + rng.normal(0, 0.05 * y_true + 0.4), 0.0)
    return U, y_obs

In [2]:
def coverages(seeds):
    epistemic, gaussian, conformal = [], [], []
    for seed in seeds:
        U, y = campaign(seed)
        tr, ca, te = slice(0, 70), slice(70, 100), slice(100, 120)
        gp = fit_gp(U[tr], y[tr], seed=seed)

        # calibration split -> one multiplier, from residuals the model never saw
        mc, sdc = gp.predict(U[ca], include_noise=True)
        q = split_conformal_multiplier(y[ca], mc, sdc, level=NOMINAL)

        m_noise, sd_noise = gp.predict(U[te], include_noise=True)
        m_epi,   sd_epi   = gp.predict(U[te], include_noise=False)
        err_noise = np.abs(m_noise - y[te])
        err_epi   = np.abs(m_epi - y[te])

        epistemic.append(np.mean(err_epi   <= GAUSS_90 * sd_epi))
        gaussian.append( np.mean(err_noise <= GAUSS_90 * sd_noise))
        conformal.append(np.mean(err_noise <= q * sd_noise))
    return epistemic, gaussian, conformal

seeds = range(6)
epistemic, gaussian, conformal = coverages(seeds)

for name, cov in (
    ("epistemic-only Gaussian", epistemic),
    ("Gaussian, noise included", gaussian),
    ("split conformal", conformal),
):
    print(f"  {name:26s} {np.mean(cov):.3f}   (nominal {NOMINAL})")

  epistemic-only Gaussian    0.583   (nominal 0.9)
  Gaussian, noise included   0.900   (nominal 0.9)
  split conformal            0.958   (nominal 0.9)


In [3]:
U, y = campaign(0)
tr, ca = slice(0, 70), slice(70, 100)
gp = fit_gp(U[tr], y[tr], seed=0)
mc, sdc = gp.predict(U[ca], include_noise=True)
q = split_conformal_multiplier(y[ca], mc, sdc, level=NOMINAL)

print(f"  Gaussian multiplier : {GAUSS_90:.3f}")
print(f"  conformal multiplier: {q:.3f}")
print(f"  intervals are {q / GAUSS_90:.2f}x wider than the Gaussian interval")

  Gaussian multiplier : 1.645
  conformal multiplier: 1.960
  intervals are 1.19x wider than the Gaussian interval


In [4]:
for label, s in (("6 seeds", range(6)), ("12 seeds", range(12))):
    epi, _, conf = coverages(s)
    print(f"  {label:9s} epistemic-only {np.mean(epi):.3f}   split conformal {np.mean(conf):.3f}")

  6 seeds   epistemic-only 0.583   split conformal 0.958


  12 seeds  epistemic-only 0.558   split conformal 0.962


In [5]:
from engin_core.gp import mapie_split_interval

te = slice(100, 120)
lo, hi = mapie_split_interval(gp, U[ca], y[ca], U[te], level=NOMINAL)
covered = np.mean((y[te] >= lo) & (y[te] <= hi))
_, sd_te = gp.predict(U[te], include_noise=True)
widths = hi - lo
print(f"  MAPIE constant-width coverage: {covered:.3f}")
print(f"  width, narrowest to widest:    {widths.min():.3f} to {widths.max():.3f} g/L")
print(f"  sd-normalized, for comparison: {(2 * q * sd_te).min():.3f} to {(2 * q * sd_te).max():.3f} g/L")

  MAPIE constant-width coverage: 0.950
  width, narrowest to widest:    15.551 to 15.551 g/L
  sd-normalized, for comparison: 14.782 to 20.816 g/L
